# CMSC 173 &middot; Machine Learning &mdash; Week 4 Lab
## Regularization: Taming Overfitting with Ridge & Lasso

A flexible model can fit your training data *too* well &mdash; memorising the noise and failing on
new data. **Regularization** fights this by adding a penalty that keeps the model's weights
small. You'll *see* overfitting, fix it with **Ridge** (built from scratch), watch coefficients
shrink, meet **Lasso** (which zeroes some out), and pick the penalty strength by validation &mdash;
the bias-variance tradeoff from week 2, back again.

**How this lab works.** Each part = a short **plain-English explainer**, a **code cell**
you run, a **line-by-line walkthrough** of what it did, and an **Answer here** box. The
code does the maths; we *graph* the results so you can see what is going on.

**NumPy + Matplotlib + scikit-learn (your first sklearn lab).** **Not graded.** About 60 minutes.

---
## Part 0 &middot; Setup

Run once. This is the first lab that imports scikit-learn.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import Lasso
rng = np.random.default_rng(173)
print('Ready.')

---
## Part 1 &middot; See overfitting happen

We take a gently curved truth and only a handful of **noisy** points. Then we fit two polynomials:
a straight line (too simple) and a wiggly degree-9 curve (too flexible). Watch what the flexible
one does between the points.

In [ ]:
def truth(x): return np.sin(1.3*x) + 0.3*x           # (1) the smooth real relationship
x = np.sort(rng.uniform(0, 5, size=12))              # (2) only 12 samples
y = truth(x) + rng.normal(0, 0.25, size=12)          # (3) with noise

grid = np.linspace(0, 5, 300)
plt.figure(figsize=(7.5,4.5))
plt.scatter(x, y, zorder=3, label='the 12 noisy points')
plt.plot(grid, truth(grid), 'k--', alpha=0.5, label='true curve')
for deg, c in [(1, 'green'), (9, 'red')]:            # (4) fit degree 1 and degree 9
    coef = np.polyfit(x, y, deg)
    plt.plot(grid, np.polyval(coef, grid), c, label=f'degree {deg} fit')
plt.ylim(-2, 4); plt.legend(); plt.title('Too simple vs too flexible')
plt.tight_layout(); plt.show()

**Reading the code, line by line:**
- **(1)&ndash;(3)** the true curve is smooth; our 12 points are that curve plus random noise.
- **(4)** `np.polyfit(x, y, deg)` fits a polynomial of the given degree. Degree 1 is a line
  (**underfits** &mdash; misses the curve). Degree 9 threads *every* dot but swings wildly between
  them (**overfits** &mdash; it has memorised the noise). Neither generalises well.

**Answer here:**

1. The degree-9 curve passes closer to the training dots than the true curve does. Why is that a
   *bad* sign, not a good one?
   &rarr; *your answer*

2. In week-2 language: does the wiggly curve have high **bias** or high **variance**?
   &rarr; *your answer*

---
## Part 2 &middot; Ridge regression, from scratch

Ridge keeps the flexibility but adds a penalty on big coefficients. Its closed form is the normal
equation with one extra term:

$$\theta = (X^\top X + \lambda I)^{-1} X^\top y.$$

The $\lambda I$ nudges every coefficient toward zero; bigger $\lambda$ = stronger pull = smoother
curve. We build degree-9 features, then fit with $\lambda = 0$ (plain, wiggly) and a larger
$\lambda$ (tamed).

In [ ]:
def design(x, degree=9):                             # (1) [1, x, x^2, ..., x^9], standardised
    P = np.vstack([x**k for k in range(1, degree+1)]).T
    P = (P - P.mean(0)) / P.std(0)                   # standardise so lambda hits fairly
    return np.column_stack([np.ones(len(x)), P])     # add the bias column

def ridge_fit(X, y, lam):                            # (2) the boxed formula
    n = X.shape[1]
    pen = lam * np.eye(n); pen[0, 0] = 0             # (3) don't penalise the intercept
    return np.linalg.solve(X.T @ X + pen, X.T @ y)

Xtr = design(x)
plt.figure(figsize=(7.5,4.5)); plt.scatter(x, y, zorder=3)
Xg = design(grid)
for lam, c in [(0.0, 'red'), (5.0, 'blue')]:         # (4) no penalty vs a real penalty
    theta = ridge_fit(Xtr, y, lam)
    plt.plot(grid, Xg @ theta, c, label=f'ridge, lambda={lam}')
plt.plot(grid, truth(grid), 'k--', alpha=0.5, label='truth')
plt.ylim(-2, 4); plt.legend(); plt.title('Ridge: a penalty smooths the fit'); plt.tight_layout(); plt.show()

**Reading the code, line by line:**
- **(1)** `design` turns one feature into columns $x, x^2, \dots, x^9$, standardises them (so the
  penalty treats them fairly), and prepends a 1s column for the intercept.
- **(2)&ndash;(3)** `ridge_fit` is the boxed formula. `pen` is $\lambda I$, but we zero its top-left
  entry so the **intercept is never penalised** (shrinking it would just shift everything down).
- **(4)** $\lambda=0$ is ordinary least squares &mdash; the wiggly red curve. $\lambda=5$ pulls the
  coefficients in and the blue curve is far smoother, much closer to the dashed truth.

**Answer here:**

1. Try `lam=100`. What happens to the curve &mdash; does it start to *underfit*? What does that tell
   you about picking lambda too large?
   &rarr; *your answer*

---
## Part 3 &middot; Watch the coefficients shrink

The whole point of Ridge is smaller coefficients. Let's sweep lambda across a wide range and plot
every coefficient's size against it &mdash; a **regularization path**.

In [ ]:
lams = np.logspace(-3, 3, 40)                        # (1) 40 lambdas from 0.001 to 1000
paths = np.array([ridge_fit(Xtr, y, l)[1:] for l in lams])   # (2) coefficients (skip intercept)

plt.figure(figsize=(7.5,4.5))
for j in range(paths.shape[1]):
    plt.plot(lams, paths[:, j])                      # (3) one line per coefficient
plt.xscale('log'); plt.xlabel('lambda (penalty strength)'); plt.ylabel('coefficient value')
plt.axhline(0, color='k', lw=0.8); plt.title('Ridge path: bigger lambda -> smaller coefficients')
plt.tight_layout(); plt.show()

**Reading the code, line by line:**
- **(1)** `np.logspace(-3, 3, 40)` gives 40 values spread evenly on a log scale (0.001 &hellip; 1000).
- **(2)** for each lambda we refit and keep the 9 coefficients (the `[1:]` drops the intercept).
- **(3)** each line is one coefficient as lambda grows. On the left (tiny lambda) they're large and
  spread out; as lambda grows they all get **squeezed toward zero** &mdash; but, for Ridge, never
  exactly to zero.

**Answer here:**

1. On the far right of the plot (huge lambda), all coefficients are near zero. What kind of model
   is that &mdash; roughly just a flat line? Which failure (over/underfitting) is that?
   &rarr; *your answer*

---
## Part 4 &middot; Lasso: same idea, sharper edge

Ridge squeezes coefficients *toward* zero. **Lasso** (penalising the sum of *absolute* values
instead of squares) can push some coefficients **exactly to zero** &mdash; automatic feature
selection. Lasso has no clean formula, so here we finally use scikit-learn.

In [ ]:
lasso = Lasso(alpha=0.05, max_iter=10000)            # (1) sklearn's Lasso, alpha = penalty
lasso.fit(Xtr[:, 1:], y)                             # (2) fit on the features (it adds its own bias)

print('Lasso coefficients (rounded):')
print(np.round(lasso.coef_, 3))
n_zero = np.sum(lasso.coef_ == 0)                    # (3) count the exact zeros
print(f'{n_zero} of {len(lasso.coef_)} coefficients were set to EXACTLY zero')

**Reading the code, line by line:**
- **(1)** `Lasso(alpha=...)` &mdash; sklearn calls the penalty strength `alpha` (our lambda). `max_iter`
  just lets its solver run long enough.
- **(2)** `.fit(features, y)` trains it; sklearn handles the intercept for you.
- **(3)** several coefficients come out as *exactly* `0`. Ridge never does that &mdash; that sparsity is
  Lasso's signature, and it's why people use Lasso to pick which features matter.

**Answer here:**

1. Ridge shrank coefficients but kept all 9; Lasso zeroed several. In one sentence, when would the
   'exactly zero' behaviour be genuinely useful (think: 500 features, only a few matter)?
   &rarr; *your answer*

---
## Part 5 &middot; Choosing lambda: the bias-variance tradeoff, again

Which lambda is best? Not the one that fits the *training* data best (that's lambda=0, the wiggle).
We hold out a **validation** set the model never trains on, and pick the lambda with the lowest
error *there*. Watch the U-shape &mdash; the same bias-variance curve you graphed in week 2.

In [ ]:
xtr, ytr = x[:8], y[:8]                              # (1) train on 8 points
xva = np.sort(rng.uniform(0, 5, 40)); yva = truth(xva) + rng.normal(0, 0.25, 40)  # (2) fresh val set

Xtr8, Xva = design(xtr), design(xva)
train_err, val_err = [], []
for l in lams:
    th = ridge_fit(Xtr8, ytr, l)
    train_err.append(np.mean((Xtr8 @ th - ytr)**2))  # (3) error on data it trained on
    val_err.append(np.mean((Xva @ th - yva)**2))     # (4) error on unseen data
best = lams[int(np.argmin(val_err))]

plt.figure(figsize=(7.5,4.5))
plt.plot(lams, train_err, 'o-', label='training error (always low)')
plt.plot(lams, val_err,   's-', label='validation error (what we care about)')
plt.axvline(best, ls='--', color='gray', label=f'best lambda ~ {best:.2g}')
plt.xscale('log'); plt.xlabel('lambda'); plt.ylabel('mean squared error')
plt.title('Pick lambda by validation error, not training error'); plt.legend()
plt.tight_layout(); plt.show()

**Reading the code, line by line:**
- **(1)&ndash;(2)** split into a small training set and a *separate* validation set the model never sees.
- **(3)** training error keeps dropping as lambda shrinks &mdash; the model just hugs its own data.
- **(4)** validation error is **U-shaped**: too-small lambda overfits (high variance), too-large
  underfits (high bias), and the dashed line marks the sweet spot in between. Same lesson as the
  week-2 bias-variance graph, now for a real model.

**Answer here:**

1. Training error was lowest at the *smallest* lambda, but validation error was not. In one
   sentence: why is it a mistake to choose a model by its training error?
   &rarr; *your answer*

2. Point to the part of the validation curve that is 'high variance' and the part that is 'high
   bias'. Which side is which?
   &rarr; *your answer*

---
## Where you actually are

Set the pace honestly. Replace each `-` with: **solid** / **rusty** / **never really got it**.

| | You |
|---|---|
| Spotting overfitting on a graph | - |
| The Ridge formula (X^T X + lambda I)^{-1} X^T y | - |
| Reading a coefficient path | - |
| How Lasso differs from Ridge | - |
| Choosing lambda by validation error | - |

**Which part took longest, and where did you get stuck?**
&rarr; *your answer*

**In one plain sentence: what does adding a penalty on the weights buy you?**
&rarr; *your answer*

---
## Stretch &mdash; optional

Required part is done; nothing below is graded.

### Stretch &middot; Ridge vs Lasso, side by side

Refit **Ridge** at `lam=1` and print its coefficients next to Lasso's from Part 4. Fill the line.

In [ ]:
# your code here: theta_ridge = ridge_fit(Xtr, y, 1.0); print its coefficients [1:]
# then eyeball: which method left more coefficients non-zero?


---
## Submitting

Run the cell below. It uploads this notebook straight from Colab &mdash; nothing to download.

You need a **submit token**: open
[https://portal.latarak.com/student/submit-token](https://portal.latarak.com/student/submit-token),
sign in, press the button, then paste it when the cell asks. The cell hides what you type.

In [ ]:
# --- Submit this notebook ------------------------------------------------------
# Colab only. Anywhere else, use the manual route described below this cell.
import getpass, json, urllib.request, urllib.error

PORTAL, COURSE, WEEK = "https://portal.latarak.com", "cmsc173", 4

try:
    from google.colab import _message
except ImportError:
    raise SystemExit(
        "Not running in Colab. Download this notebook "
        "(File > Download > Download .ipynb) and upload it at "
        "https://portal.latarak.com/course/cmsc173/lab/4/submit"
    )

nb = _message.blocking_request("get_ipynb", timeout_sec=90)["ipynb"]
token = getpass.getpass("Submit token (hidden as you type): ").strip()

req = urllib.request.Request(
    PORTAL + "/api/labs/" + COURSE + "/submit-notebook",
    data=json.dumps({"week": WEEK, "notebook": nb}).encode(),
    headers={"Content-Type": "application/json", "Authorization": "Bearer " + token},
    method="POST",
)
try:
    with urllib.request.urlopen(req, timeout=120) as r:
        out = json.load(r)
    print("Submitted", out["course"], "week", out["week"], "for", out["student"])
    print(out["cells"], "cells,", out["executed"], "executed")
    print(out["message"])
except urllib.error.HTTPError as e:
    print("Not submitted:", json.loads(e.read()).get("error", e.reason))

Prefer to do it by hand? **File &rarr; Download &rarr; Download .ipynb**, then go to the
[Week 4 submission page](https://portal.latarak.com/course/cmsc173/lab/4/submit) and upload it.

Blank cells are fine and guesses are fine. Don't polish this until it hides what you knew.